In [1]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

import pandas as pd
import openai

In [14]:
# Set up OpenAI API key
from dotenv import load_dotenv
import os
from pathlib import Path

# Load environment variables from .env file in the project root
env_path = Path("../../.env")  # Adjust path if needed
if env_path.exists():
    load_dotenv(env_path, override=True)
else:
    # Try alternative path
    env_path = Path(".env")
    if env_path.exists():
        load_dotenv(env_path, override=True)

# Initialize OpenAI client with API key
# Option 1: Using environment variable (recommended)
openai_client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Option 2: If you prefer to set it directly (less secure, but simple for testing)
# openai_client = openai.OpenAI(api_key="your-api-key-here")

# Verify the API key is set
if os.getenv("OPENAI_API_KEY"):
    print("✓ OpenAI API key loaded from environment")
else:
    print("⚠ Warning: OPENAI_API_KEY not found in environment variables")
    print("  Make sure your .env file contains: OPENAI_API_KEY=your-key-here")

✓ OpenAI API key loaded from environment


In [114]:



df_rating_sample_2000 = pd.read_json('../../Data/Movies_and_TV.jsonl/Movies_and_TV_ratings_sample_2000.jsonl',lines=True)
df_sample_2000 = pd.read_json('../../Data/sample_2000.jsonl',lines=True)


In [115]:
df_sample_2000



,main_category,title,subtitle,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
0,Prime Video,None,None,4.7,65,None,None,NaN,[{'360w': 'https://m.media-amazon.com/images/G...,[],NaN,None,"{'Content advisory': ['Violence', 'alcohol use...",B01LYA41BY,NaN
1,Prime Video,None,None,4.6,97881,None,None,NaN,[{'360w': 'https://images-na.ssl-images-amazon...,[],NaN,None,"{'Content advisory': ['Violence', 'alcohol use...",B00ZHWD78I,NaN
2,Prime Video,None,None,4.9,18911,None,None,NaN,[{'360w': 'https://images-na.ssl-images-amazon...,[],NaN,None,"{'Content advisory': ['Violence', 'substance u...",B00U7Q0CXC,NaN
3,Prime Video,None,None,4.6,1346,None,None,NaN,[{'360w': 'https://images-na.ssl-images-amazon...,[],NaN,None,"{'Content advisory': ['Violence', 'alcohol use...",B01HUGA1AU,NaN
4,Prime Video,None,None,4.6,114,None,None,NaN,[{'360w': 'https://images-na.ssl-images-amazon...,[],NaN,None,"{'Content advisory': ['Violence', 'smoking', '...",B00TE84WCG,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,Prime Video,None,None,4.7,637,None,None,NaN,[{'360w': 'https://images-na.ssl-images-amazon...,[],NaN,None,"{'Content advisory': ['Nudity', 'violence', 'f...",B000OZNHR4,NaN
1996,Prime Video,None,None,4.9,625,None,None,NaN,[{'360w': 'https://m.media-amazon.com/images/G...,[],NaN,None,"{'Audio languages': ['English'], 'Subtitles': ...",B008RKKSLG,NaN
1997,Prime Video,None,None,4.7,36,None,None,NaN,[{'360w': 'https://m.media-amazon.com/images/G...,[],NaN,None,"{'Audio languages': ['English'], 'Subtitles': ...",B073RPT5CL,NaN
1998,Prime Video,None,None,4.6,13377,None,None,NaN,[{'360w': 'https://images-na.ssl-images-amazon...,[],NaN,None,"{'Content advisory': ['Nudity', 'violence', 'a...",B00GIXY4M8,NaN


In [117]:
df_rating_sample_2000.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5,I'm a sucker for these sappy movies,"I like any movie Hilarie Burton is in, I reall...",[],B07FN83ZK1,B07FN83ZK1,AFGNIWCBLQT2QIXXKIW7Q6VREZRQ,2020-10-01 02:00:01.603,0,True
1,5,Funny,I love this show. It is funny and keeps me lau...,[],B0754PSVL2,B0754PSVL2,AFUOYIZBU3MTBOLYKOJE5Z35MBDA,2018-04-27 16:11:35.227,0,False
2,5,Encanto,One of my favorite Disneys to date.. Encanto. ...,[],B0B12GSRKP,B0B12GSRKP,AFZUK3MTBIBEDQOPAK3OATUOUKLA,2022-11-26 23:08:35.853,0,True
3,5,the hundred foot journey,One of my husband's favorites. He loves stori...,[],B00QGKWI08,B00QGKWI08,AFZUK3MTBIBEDQOPAK3OATUOUKLA,2019-05-18 03:09:12.106,0,True
4,5,Loved this kid-friendly movie,"We loved this movie as a family. Funny, witho...",[],B00ENYKO82,B00ENYKO82,AFMOHJOF54QARPV5DJNPJDN3F4AQ,2013-09-01 00:24:08.000,0,True


In [92]:
list(df_rating_sample_2000["text"].items())[0]
#list(df_rating_sample_2000["title"].items())[0]

(0,
 "I like any movie Hilarie Burton is in, I really like her. This movie is like most  Hallmark movies and they are my guilty pleasure. I liked seeing all the food in the movie and love how the characters come together. I've watched this movie several times and I am sure I'll watch it several more. I like movies that have happy endings and Hallmark certainly is the place to go for that.")

In [119]:
def preprocess_description(row):
    return f"{row['title']} {' '.join(row['text'])}"

In [120]:

df_rating_sample_2000["description"] = df_rating_sample_2000.apply(preprocess_description, axis=1)

In [121]:
df_rating_sample_2000.head(2)

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,description
0,5,I'm a sucker for these sappy movies,"I like any movie Hilarie Burton is in, I reall...",[],B07FN83ZK1,B07FN83ZK1,AFGNIWCBLQT2QIXXKIW7Q6VREZRQ,2020-10-01 02:00:01.603,0,True,I'm a sucker for these sappy movies I l i k ...
1,5,Funny,I love this show. It is funny and keeps me lau...,[],B0754PSVL2,B0754PSVL2,AFUOYIZBU3MTBOLYKOJE5Z35MBDA,2018-04-27 16:11:35.227,0,False,Funny I l o v e t h i s s h o w . I t ...


In [113]:
df_rating_sample_2000.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,description
0,5,I'm a sucker for these sappy movies,"I like any movie Hilarie Burton is in, I reall...",[],B07FN83ZK1,B07FN83ZK1,AFGNIWCBLQT2QIXXKIW7Q6VREZRQ,2020-10-01 02:00:01.603,0,True,I'm a sucker for these sappy movies I l i k ...
1,5,Funny,I love this show. It is funny and keeps me lau...,[],B0754PSVL2,B0754PSVL2,AFUOYIZBU3MTBOLYKOJE5Z35MBDA,2018-04-27 16:11:35.227,0,False,Funny I l o v e t h i s s h o w . I t ...
2,5,Encanto,One of my favorite Disneys to date.. Encanto. ...,[],B0B12GSRKP,B0B12GSRKP,AFZUK3MTBIBEDQOPAK3OATUOUKLA,2022-11-26 23:08:35.853,0,True,Encanto O n e o f m y f a v o r i t e ...
3,5,the hundred foot journey,One of my husband's favorites. He loves stori...,[],B00QGKWI08,B00QGKWI08,AFZUK3MTBIBEDQOPAK3OATUOUKLA,2019-05-18 03:09:12.106,0,True,the hundred foot journey O n e o f m y h...
4,5,Loved this kid-friendly movie,"We loved this movie as a family. Funny, witho...",[],B00ENYKO82,B00ENYKO82,AFMOHJOF54QARPV5DJNPJDN3F4AQ,2013-09-01 00:24:08.000,0,True,Loved this kid-friendly movie W e l o v e d ...


In [122]:
def preprocess_description(row):
    return f"{row['title']} {' '.join(row['features'])}"

In [ ]:
def extract_first_large_image(row):
    return row["images"][0].get("large", "")

In [ ]:

df_rating_sample_2000["description"] = df_rating_sample_2000.apply(preprocess_description, axis=1)


In [97]:
list(df_rating_sample_2000["description"].items())[0]

(0,
 "I'm a sucker for these sappy movies I   l i k e   a n y   m o v i e   H i l a r i e   B u r t o n   i s   i n ,   I   r e a l l y   l i k e   h e r .   T h i s   m o v i e   i s   l i k e   m o s t     H a l l m a r k   m o v i e s   a n d   t h e y   a r e   m y   g u i l t y   p l e a s u r e .   I   l i k e d   s e e i n g   a l l   t h e   f o o d   i n   t h e   m o v i e   a n d   l o v e   h o w   t h e   c h a r a c t e r s   c o m e   t o g e t h e r .   I ' v e   w a t c h e d   t h i s   m o v i e   s e v e r a l   t i m e s   a n d   I   a m   s u r e   I ' l l   w a t c h   i t   s e v e r a l   m o r e .   I   l i k e   m o v i e s   t h a t   h a v e   h a p p y   e n d i n g s   a n d   H a l l m a r k   c e r t a i n l y   i s   t h e   p l a c e   t o   g o   f o r   t h a t .")

In [ ]:
from openai import OpenAI

client = OpenAI(api_key="")

response = client.embeddings.create(
    model="text-embedding-3-small",
    input="Random text",
)



In [20]:
response

CreateEmbeddingResponse(data=[Embedding(embedding=[-0.015960441902279854, -0.0039901104755699635, 0.008389661088585854, 0.0036115716211497784, -0.042859893292188644, -0.0627911314368248, -0.007883654907345772, 0.01200509537011385, -0.013055735267698765, -0.006126152351498604, -0.006767351180315018, 0.011781062930822372, -0.02079647220671177, 0.03779210150241852, 0.02839815430343151, -0.01854068785905838, -0.032044488936662674, -0.006666922476142645, -0.05052337795495987, 0.03371315076947212, -0.015489200130105019, 0.024365555495023727, -0.020178448408842087, 4.9278882215730846e-05, 0.019189612939953804, -0.0015257442137226462, 0.01586773991584778, -0.008714123629033566, 0.036370649933815, -0.036370649933815, 0.024489158764481544, -0.04298349469900131, 0.022418782114982605, -0.025400742888450623, 0.013936417177319527, 0.009108113124966621, 0.05036887153983116, 7.27866863599047e-05, 0.022480584681034088, 0.0022480585612356663, -0.0031654362101107836, -0.04743326082825661, 0.0315809734165

In [19]:
type(response)

openai.types.create_embedding_response.CreateEmbeddingResponse

In [21]:
len(response.data[0].embedding)

1536

In [143]:
def get_embedding(text, model="text-embedding-3-small"):
    # Use the openai_client instance created in Cell 1
    response = openai_client.embeddings.create(
        input=text,
        model=model,
    )
    return response.data[0].embedding

In [ ]:
get_embedding("Hello, world!")

In [106]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [ ]:
qdrant_client.create_collection(
    collection_name="Amazon-items-collection-00",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

Create pointstruct to structure the embedded vectors for loading into quadrant collection

In [108]:
pointstruct = PointStruct(
    id=0,
    vector=get_embedding("Test text"),
    payload={
        "text": "Test text",
        "model": "text-embedding-3-small",
    },
)

In [128]:
data_to_embed = df_rating_sample_2000[["description","rating","asin","parent_asin"]].to_dict(orient="records")

In [137]:
data_to_embed_dev = data_to_embed[0:9]

In [139]:
data_to_embed_dev

[{'description': "I'm a sucker for these sappy movies I   l i k e   a n y   m o v i e   H i l a r i e   B u r t o n   i s   i n ,   I   r e a l l y   l i k e   h e r .   T h i s   m o v i e   i s   l i k e   m o s t     H a l l m a r k   m o v i e s   a n d   t h e y   a r e   m y   g u i l t y   p l e a s u r e .   I   l i k e d   s e e i n g   a l l   t h e   f o o d   i n   t h e   m o v i e   a n d   l o v e   h o w   t h e   c h a r a c t e r s   c o m e   t o g e t h e r .   I ' v e   w a t c h e d   t h i s   m o v i e   s e v e r a l   t i m e s   a n d   I   a m   s u r e   I ' l l   w a t c h   i t   s e v e r a l   m o r e .   I   l i k e   m o v i e s   t h a t   h a v e   h a p p y   e n d i n g s   a n d   H a l l m a r k   c e r t a i n l y   i s   t h e   p l a c e   t o   g o   f o r   t h a t .",
  'rating': 5,
  'asin': 'B07FN83ZK1',
  'parent_asin': 'B07FN83ZK1'},
 {'description': 'Funny I   l o v e   t h i s   s h o w .   I t   i s   f u n n y   a n d   k e e p s  

In [ ]:
pointstructs = []
for i, data in enumerate(data_to_embed_dev):
    embeding = get_embedding(data["description"])
    pointstructs.append(
        PointStruct(
            id=i,
            vector=embeding,
            payload=data,
        )
    )

In [136]:
pointstructs

[PointStruct(id=0, vector=[-0.009013412520289421, -0.013456806540489197, -0.0641414225101471, 0.00398293836042285, 0.041256796568632126, -0.03614573925733566, 0.04252304509282112, 0.022159410640597343, 0.003381468588486314, -0.028179863467812538, 0.03573133051395416, -0.044986482709646225, -0.043282799422740936, 0.019247030839323997, -0.01816496066749096, 0.05705041065812111, 0.0289626382291317, -0.02725895307958126, 0.051939357072114944, -0.04413463920354843, 0.050650082528591156, 0.054794181138277054, -0.0471506230533123, 0.009859499521553516, 0.03810267522931099, -0.013387737795710564, -0.02647618018090725, 0.06018150597810745, 0.06068800762295723, 0.022608354687690735, -0.0002061256964225322, -0.022907651960849762, 0.018521813675761223, -0.0628521516919136, -0.01065378449857235, -0.01063076127320528, -0.016438255086541176, 0.009698339737951756, -0.0018259930657222867, -0.03715874254703522, 0.006014697253704071, 0.04144097864627838, 0.06814738363027573, -0.030873527750372887, 0.0433

Wtite embedded data to quadrant client

In [135]:
qdrant_client.upsert(
    collection_name="Amazon-items-collection-00",
    wait=True,
    points=pointstructs,
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

Define query embedding function

In [ ]:
def retrieve_data(query, k=5):
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-00",
        query=query_embedding,
        limit=k,
    )
    return results

retrieved_context = retrieve_data("What kind of earphones can I get?", qdrant_client, k=10)

In [142]:
retrieve_data("what movies are recommended for children?").points

[ScoredPoint(id=4, version=1, score=0.50967324, payload={'description': 'Loved this kid-friendly movie W e   l o v e d   t h i s   m o v i e   a s   a   f a m i l y .     F u n n y ,   w i t h o u t   a l l   o f   t h e   c r u d e   h u m o r   t h a t   t e n d s   t o   g o   o n   w i t h   e v e n   & # 3 4 ; k i d - f r i e n d l y & # 3 4 ;   m o v i e s .     O n e   o f   o u r   f a v e s   f o r   s u r e !', 'rating': 5, 'asin': 'B00ENYKO82', 'parent_asin': 'B00ENYKO82'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=6, version=1, score=0.28082705, payload={'description': 'great movie. I like how the movie was presented ... g r e a t   m o v i e . I   l i k e   h o w   t h e   m o v i e   w a s   p r e s e n t e d   w i t h   s h o w i n g   t h e   p i l o t   a f t e r   t h e   c r a s h   t h e n   e x p l a i n i n g   t h e   s t o r y .', 'rating': 5, 'asin': 'B01LW32XQV', 'parent_asin': 'B01LW32XQV'}, vector=None, shard_key=None, order_value=None)